In [1]:
import ast
import re

from datasets import load_dataset
from utils import flatten_list

# Arithmetic dataset

In [2]:
dataset = load_dataset("baharef/ToT", "tot_arithmetic")

In [3]:
df = dataset["test"].to_pandas()

In [4]:
df.question_type.unique()

array(['add_subtract', 'compare', 'duration', 'multi_op', 'schedule',
       'trick', 'timezone'], dtype=object)

In [5]:
df = df.assign(label_as_dict=lambda x: x["label"].apply(ast.literal_eval))

## Add Substract

In [6]:
add_sub = df.query("question_type == 'add_subtract'").assign(
    label_keys=lambda x: x["label_as_dict"].apply(lambda y: tuple(y.keys())),
    label_values=lambda x: x["label_as_dict"].apply(lambda y: tuple(y.values())),
)
add_sub.loc[:, "label_keys"].unique()

array([('answer',), ('date',)], dtype=object)

In [7]:
add_sub.loc[:, "label_values"].apply(lambda x: x[0]).str.replace(
    r"\d\d\/\d\d/\d\d\d\d", "dd/mm/yyyy", regex=True
).str.replace(r"\d+ [ABCD]{2}", "BC/AD", regex=True).str.replace(
    r"\d{4}", "YYYY", regex=True
).unique()

array(['BC/AD', 'Thursday', 'Sunday', 'Saturday', 'Wednesday', 'Tuesday',
       'Friday', 'Monday', 'dd/mm/yyyy', 'YYYY'], dtype=object)

## Compare

In [8]:
compare = df.query("question_type == 'compare'").assign(
    label_keys=lambda x: x["label_as_dict"].apply(lambda y: tuple(y.keys())),
    label_values=lambda x: x["label_as_dict"].apply(lambda y: tuple(y.values())),
)
compare.label_keys.unique()

array([('unordered_list',), ('answer',), ('ordered_list',)], dtype=object)

In [9]:
unique_answeres = set(flatten_list(
    df.query("question_type == 'compare' and label.str.contains('unordered_list')")
    .loc[:, "label_as_dict"]
    .apply(lambda x: tuple(x["unordered_list"]))
    .unique()
))
[a for a in unique_answeres if re.match(r"\d+", a)]

[]

In [10]:
unique_answeres = set(
    flatten_list(
        df.query(
            "question_type == 'compare' and label.str.contains('{\\'ordered_list\\':')",
        )
        .loc[:, "label_as_dict"]
        .apply(lambda x: tuple(x["ordered_list"]))
        .unique()
    )
)
[a for a in unique_answeres if re.match(r"\d+", a)]

[]

The compare subset asking for an ordered and unordered list do not contain numbers (dates) as its expected answer.

In [11]:
unique_answeres = (
    df.query(
        "question_type == 'compare' and label.str.contains('answer')",
    )
    .loc[:, "label_as_dict"]
    .apply(lambda x: x["answer"])
    .unique()
)

unique_answeres
# [a for a in unique_answeres if re.match(r"\d+", a)]

array(['E49', 'E47', 'E33', 'E2', 'E14', 'E46', 'E5', 'E15', 'E45', 'E4',
       'E31', 'E1', 'E41', 'E40', 'E34', 'E3', 'E28', 'E20', 'E26', 'E22',
       'E35', 'E48', 'E13', 'E32', 'E18', 'E30', 'E12', 'E29', 'E37',
       'E17', 'E10'], dtype=object)

The compare subset asking for an answer is expecting an entityt, not a date

## Duration

In [12]:
duration = df.query("question_type == 'duration'").assign(
    label_keys=lambda x: x["label_as_dict"].apply(lambda y: tuple(y.keys())),
    label_values=lambda x: x["label_as_dict"].apply(lambda y: tuple(y.values())),
)
duration.label_keys.unique()

array([('answer',), ('hours', 'minutes', 'seconds')], dtype=object)

In [13]:
duration.query("label.str.contains('hours')").loc[:, "label"].str.replace("\d+", "<D>", regex=True).unique()

array(["{'hours': <D>, 'minutes': <D>, 'seconds': <D>}"], dtype=object)

In [14]:
duration.query("label.str.contains('answer')").loc[:, "label"].str.replace("\d+", "<D>", regex=True).unique()

array(["{'answer': '<D>'}", "{'answer': <D>}"], dtype=object)

## Multi OP

In [15]:
df.query("question_type == 'multi_op'").assign(
    label_keys=lambda x: x["label_as_dict"].apply(lambda y: tuple(y.keys()))
).label_keys.unique()

array([('H', 'M', 'S'), ('answer',), ('A', 'B', 'C'), ('day', 'time'),
       ('X', 'Y', 'Z')], dtype=object)

In [16]:
df.query("question_type == 'multi_op' and label.str.startswith('{\"A')").head()

,question_type,question,label,label_as_dict
1050,multi_op,Emma has created a robot to do her tasks. It t...,"{""A"": 80, ""B"": 22, ""C"": 20}","{'A': 80, 'B': 22, 'C': 20}"
1051,multi_op,Zoe has created a robot to do her tasks. It ta...,"{""A"": 15, ""B"": 31, ""C"": 48}","{'A': 15, 'B': 31, 'C': 48}"
1052,multi_op,Zoe has bought a robot that helps with everyda...,"{""A"": 39, ""B"": 48, ""C"": 56}","{'A': 39, 'B': 48, 'C': 56}"
1053,multi_op,Mia has bought a robot that helps with everyda...,"{""A"": 42, ""B"": 48, ""C"": 0}","{'A': 42, 'B': 48, 'C': 0}"
1054,multi_op,Stella has created a robot to do her tasks. It...,"{""A"": 78, ""B"": 26, ""C"": 24}","{'A': 78, 'B': 26, 'C': 24}"


In [17]:
df.query("question_type == 'multi_op' and label.str.startswith('{\"X')").head()

,question_type,question,label,label_as_dict
1150,multi_op,It takes Sarah an average of 22 minutes and 16...,"{""X"": 2.0, ""Y"": 24.0, ""Z"": 44.0}","{'X': 2.0, 'Y': 24.0, 'Z': 44.0}"
1151,multi_op,It takes Nora an average of 21 minutes and 56 ...,"{""X"": 2.0, ""Y"": 22.0, ""Z"": 34.0}","{'X': 2.0, 'Y': 22.0, 'Z': 34.0}"
1152,multi_op,It takes Hannah an average of 30 minutes and 4...,"{""X"": 4.0, ""Y"": 51.0, ""Z"": 39.0}","{'X': 4.0, 'Y': 51.0, 'Z': 39.0}"
1153,multi_op,It takes Natalie an average of 13 minutes and ...,"{""X"": 2.0, ""Y"": 2.0, ""Z"": 6.0}","{'X': 2.0, 'Y': 2.0, 'Z': 6.0}"
1154,multi_op,It takes Camila an average of 22 minutes and 5...,"{""X"": 3.0, ""Y"": 48.0, ""Z"": 20.0}","{'X': 3.0, 'Y': 48.0, 'Z': 20.0}"


In [18]:
df.query("question_type == 'multi_op' and label.str.contains('answer')").loc[
    :, "label_as_dict"
].apply(lambda x: x["answer"])

950          24 May, 2023
951     November 11, 2010
952            07-27-2002
953            09-04-2014
954            12-02-2011
955       August 22, 2020
956          17 Jul, 2007
957          Oct 17, 1997
958     10 December, 2014
959            01-19-2020
960            03-08-2015
961          19 Jun, 2010
962            01-09-2016
963      October 18, 2013
964        02 March, 2020
965          Feb 28, 2010
966          21 Oct, 2005
967          Dec 22, 2004
968     15 December, 2004
969          May 23, 2016
970          29 Nov, 2011
971         03 July, 2024
972      01 October, 2016
973    28 September, 2012
974     February 25, 2005
975          Nov 23, 2019
976            02-14-2012
977            02-16-2007
978     November 01, 2017
979          07 May, 2015
980      03 October, 2000
981          23 Dec, 2020
982          Dec 12, 2011
983            30-07-1998
984          24 Dec, 2001
985          May 19, 2023
986            03-28-2020
987            06-01-2021
988         

Multi OP questions still ask for some measure of time

## Trick

In [19]:
df.query("question_type == 'trick'").assign(
    label_keys=lambda x: x["label_as_dict"].apply(lambda y: tuple(y.keys()))
).label_keys.unique()

array([('answer',), ('unordered_list',)], dtype=object)

In [20]:
df.query("question_type == 'trick' and label.str.contains('unordered_list')")

,question_type,question,label,label_as_dict
1650,trick,"On Thursday before noon, I received a text say...","{'unordered_list': ['Thursday', 'Friday']}","{'unordered_list': ['Thursday', 'Friday']}"
1651,trick,"If right now it is Friday before noon, what da...",{'unordered_list': ['Saturday']},{'unordered_list': ['Saturday']}
1652,trick,"If right now it is Monday before noon, what da...","{'unordered_list': ['Monday', 'Tuesday']}","{'unordered_list': ['Monday', 'Tuesday']}"
1653,trick,"On Wednesday before noon, I received a text sa...",{'unordered_list': ['Thursday']},{'unordered_list': ['Thursday']}
1654,trick,"On a Thursday before noon, Fred started studyi...","{'unordered_list': ['Thursday', 'Friday']}","{'unordered_list': ['Thursday', 'Friday']}"
1655,trick,"On a Saturday before noon, Fred started studyi...",{'unordered_list': ['Saturday']},{'unordered_list': ['Saturday']}
1656,trick,"On a Thursday before noon, Fred started studyi...",{'unordered_list': ['Friday']},{'unordered_list': ['Friday']}
1657,trick,"If right now it is Monday before noon, what da...","{'unordered_list': ['Monday', 'Tuesday']}","{'unordered_list': ['Monday', 'Tuesday']}"
1658,trick,"On Tuesday before noon, I received a text sayi...",{'unordered_list': ['Wednesday']},{'unordered_list': ['Wednesday']}
1659,trick,"On Friday before noon, I received a text sayin...","{'unordered_list': ['Friday', 'Saturday']}","{'unordered_list': ['Friday', 'Saturday']}"


In [21]:
df.query("question_type == 'trick' and label.str.contains('answer')").loc[:, "label_as_dict"].apply(
    lambda x: x["answer"]
).str.replace(r"\d\d\d\d-\d\d-\d\d", "YYYY-MM-DD", regex=True).unique()

array(['YYYY-MM-DD', 'Sunday', 'Wednesday', '30', 'Christina', '52',
       'unanswerable', 'Liam', 'Thursday', 'Mia', 'Tuesday', 'Brad',
       '170', '53', '21', 'Levi'], dtype=object)

## Timezone

In [22]:
df.query("question_type == 'timezone'").assign(
    label_keys=lambda x: x["label_as_dict"].apply(lambda y: tuple(y.keys()))
).label_keys.unique()

array([('hours', 'minutes'), ('day', 'time'),
       ('days', 'hours', 'minutes', 'seconds')], dtype=object)

## Number of QA pairs useful for TEA

In [23]:
weekdays = ["Thursday", "Sunday", "Saturday", "Wednesday", "Tuesday", "Friday", "Monday"]

In [24]:
# Note.
tea_subset = (
    df.query("question_type!='compare'")
    .query("~label.str.contains('|'.join(@weekdays))")
)
tea_subset.shape

(1342, 4)

In [25]:
tea_subset.loc[:, "question"].str.len().sum() / 1500

325.94733333333335

In [26]:
tea_subset.loc[:, "label"].str.len().sum() / 1500

21.318

# Semantic dataset

In [27]:
dataset_sem = load_dataset("baharef/ToT", "tot_semantic_large")

In [28]:
df_sem = dataset_sem["test"].to_pandas()

In [29]:
df_sem.question_type.unique()

array(['before_after', 'event_at_time_t', 'event_at_what_time',
       'first_last', 'event_at_the_time_of_another_event',
       'number_of_events_in_time_interval', 'relation_duration',
       'timeline'], dtype=object)

## Question types not useful for TEA

In [30]:
df_sem.query("question_type=='before_after'").label.str.replace("E\d+", "", regex=True).unique()

array([''], dtype=object)

In [31]:
df_sem.query("question_type=='event_at_time_t'").label.str.replace("E\d+", "", regex=True).unique()

array([''], dtype=object)

In [32]:
df_sem.query("question_type=='first_last'").label.str.replace("E\d+", "", regex=True).unique()

array([''], dtype=object)

In [33]:
df_sem.query("question_type=='event_at_the_time_of_another_event'").label.str.replace("E\d+", "", regex=True).unique()

array([''], dtype=object)

In [34]:
df_sem.query("question_type=='number_of_events_in_time_interval'").label.unique()

array(['0', '1', '4', '2', '3', '6', '5', '7', '9', '19'], dtype=object)

## Questions useful for TEA

In [35]:
df_sem.query("question_type=='event_at_what_time'").label.unique()

array(['1969', '1986', '1970', '1920', '1944', '1979', '1950', '2002',
       '1994', '1951', '1945', '1984', '1996', '1960', '1949', '1966',
       '1943', '2003', '1916', '1938', '1985', '1955', '1931', '1957',
       '1933', '1941', '1930', '2004', '1911', '1962', '1991', '1987',
       '1947', '2016', '1999', '1974', '1997', '1928', '1948', '1961',
       '1998', '1922', '1965', '1964', '1992', '1939', '1977', '1976',
       '2000', '1918', '1946', '1934', '1993', '1942', '1978', '1968',
       '1973', '1967', '1954', '1963', '1983', '1940', '1936', '1981',
       '1956', '1975', '2015', '2018', '1995', '1932', '1971', '1917',
       '2007', '1929', '2014', '1990', '1952', '2001', '1919', '1958',
       '1921', '1982', '1972', '1953', '1989', '1959', '1937', '2006',
       '1988', '1925', '2009', '1924', '1935', '1910', '1914', '1909',
       '1908', '1980', '1903', '1915', '1923', '2008', '2011', '2005',
       '1907', '1926', '1913', '2019', '1927', '2010', '1905', '2012',
      

In [36]:
df_sem.query("question_type=='relation_duration'").label.unique()

array(['1', '5', '6', '8', '2', '3', '7', '9', '4', '10', '0', '32', '12',
       '13', '19', '18', '11', '15'], dtype=object)

In [37]:
df_sem.query("question_type=='relation_duration'").head()

,question_type,sorting_type,graph_gen_algorithm,prompt,question,label
4980,relation_duration,target_and_start_time,er,Here is a set of temporal facts:\nE37 was the ...,For how many years did it last when E92 was th...,1
4981,relation_duration,start_time_and_target,er,Here is a set of temporal facts:\nE37 was the ...,For how many years did it last when E92 was th...,1
4982,relation_duration,shuffle,er,Here is a set of temporal facts:\nE94 was the ...,For how many years did it last when E92 was th...,1
4983,relation_duration,relation_and_start_time,er,Here is a set of temporal facts:\nE94 was the ...,For how many years did it last when E92 was th...,1
4984,relation_duration,start_time_and_relation,er,Here is a set of temporal facts:\nE94 was the ...,For how many years did it last when E92 was th...,1


## Number of QA pairs useful for TEA

In [38]:
tea_subset_sem = df_sem.query("question_type.isin(['relation_duration', 'event_at_what_time'])")

In [39]:
tea_subset_sem.shape

(11620, 6)

In [40]:
tea_subset_sem.loc[:, "question"].str.len().sum() / 11620

104.37736660929431

In [41]:
tea_subset_sem.loc[:, "label"].str.len().sum() / 11620

2.5206540447504304